# 02 — Forecasting: Budget Functions

Forecast quarterly federal obligated spending at the budget function level using **Prophet**, **SARIMA**, and **XGBoost**.

- **Train**: FY2017–2022 (24 quarters)
- **Test**: FY2023–2024 (8 quarters)
- **Horizon**: 4 quarters ahead (1 year)
- **COVID handling**: Binary flag regressor (1 for FY2020–2021)
- **Data note**: USASpending.gov reports cumulative YTD amounts per quarter — Q4 represents the full fiscal year total

**Key results:**
| Model   | MAE     | RMSE    | MAPE (Total) |
|---------|---------|---------|--------------|
| SARIMA  | $144B   | $157B   | **3.8%** — best total series |
| XGBoost | $281B   | $335B   | 8.3% |
| Prophet | $1,765B | $2,188B | 27.6% — best per stable function |

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False})

ROOT      = Path('..').resolve()
CLEAN_A   = ROOT / 'pipeline-a-hierarchical' / 'data' / 'cleaned'
FORECAST_DIR = ROOT / 'pipeline-a-hierarchical' / 'data' / 'forecasts'
FORECAST_DIR.mkdir(exist_ok=True)

def trillions(x, _): return f'${x/1e12:.2f}T'
def billions(x, _):  return f'${x/1e9:.1f}B'

def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mp   = mape(np.array(y_true), np.array(y_pred))
    print(f'{label:12}  MAE=${mae/1e9:.2f}B  RMSE=${rmse/1e9:.2f}B  MAPE={mp:.1f}%')
    return {'model': label, 'MAE': mae, 'RMSE': rmse, 'MAPE': mp}

print('Setup done.')

Prophet, SARIMAX, and XGBoost are the three model families. The `metrics()` helper prints MAE, RMSE, and MAPE in a consistent format so all three models can be compared side by side. MAE and RMSE are reported in billions of dollars; MAPE as a percentage. The `FORECAST_DIR` folder is where all predictions get saved so the dashboard can load them without re-running models.

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv(CLEAN_A / 'budget_functions_quarterly_all.csv',
                 dtype={'budget_function_id': str})

# Drop unreported rows (null function_id)
df = df.dropna(subset=['budget_function_id']).copy()

# Convert fy + quarter → actual date (federal fiscal year calendar)
# Q1=Oct, Q2=Jan, Q3=Apr, Q4=Jul
quarter_to_month = {1: (10, -1), 2: (1, 0), 3: (4, 0), 4: (7, 0)}
def fy_q_to_date(row):
    month, yr_offset = quarter_to_month[row['quarter']]
    return pd.Timestamp(year=int(row['fy']) + yr_offset, month=month, day=1)

df['ds'] = df.apply(fy_q_to_date, axis=1)

# COVID anomaly flag
df['covid'] = df['fy'].isin([2020, 2021]).astype(int)

# Quarter cyclical encoding for XGBoost
df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

print(f'Rows: {len(df):,}  FY: {df["fy"].min()}–{df["fy"].max()}  Functions: {df["budget_function_id"].nunique()}')
print(f'Date range: {df["ds"].min().date()} → {df["ds"].max().date()}')
print(df[['fy','quarter','ds','covid','budget_function_id','budget_function_name','obligated_amount']].head(6).to_string(index=False))

Federal fiscal year dates are offset from the calendar year — Q1 starts in October of the *prior* calendar year. So FY2020 Q1 = October 2019, not January 2020. Getting this right matters for Prophet since it builds its seasonality model from actual calendar dates. The COVID flag marks FY2020 and FY2021 as anomalous — it gets passed as an external regressor to all three models so they know those years were exceptional.

In [ ]:
# Build total spending series (sum across all functions per quarter)
total = (df.groupby(['fy', 'quarter', 'ds', 'covid'])
           .agg(obligated_amount=('obligated_amount', 'sum'))
           .reset_index().sort_values('ds'))

# Top 10 functions by total spend (for individual function models)
top10_ids = df.groupby('budget_function_id')['obligated_amount'].sum().nlargest(10).index.tolist()
func_names = df.drop_duplicates('budget_function_id').set_index('budget_function_id')['budget_function_name'].to_dict()

# Train / test split
TRAIN_END = 2022
TEST_START = 2023

train_total = total[total['fy'] <= TRAIN_END]
test_total  = total[total['fy'] >= TEST_START]

print(f'Total series — Train: {len(train_total)} rows  Test: {len(test_total)} rows')
print(f'Train: {train_total["ds"].min().date()} → {train_total["ds"].max().date()}')
print(f'Test:  {test_total["ds"].min().date()} → {test_total["ds"].max().date()}')
print(f'\nTop 10 functions: {top10_ids}')

Two targets are prepared here: the **total** series (all functions summed per quarter — 32 data points) and **individual function** series for the top 10. The total is the most stable series and the best one to validate the modeling approach before applying it to individual functions. The train/test split is clean: everything up to and including FY2022 is training data, FY2023 and FY2024 are held out as the test set.

## 3. Prophet

In [ ]:
def run_prophet(train_df, test_df, series_name='total'):
    prophet_train = train_df.rename(columns={'obligated_amount': 'y'})[['ds','y','covid']]
    prophet_test  = test_df.rename(columns={'obligated_amount': 'y'})[['ds','y','covid']]

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.05
    )
    m.add_regressor('covid')
    m.fit(prophet_train)

    future = m.make_future_dataframe(periods=len(test_df), freq='QS-OCT')
    future['covid'] = future['ds'].dt.year.isin([2019, 2020]).astype(int)
    forecast = m.predict(future)

    pred = forecast.tail(len(test_df))['yhat'].values
    true = test_df['obligated_amount'].values
    result = metrics(true, pred, label='Prophet')
    result['series'] = series_name
    result['pred']   = pred
    result['true']   = true
    result['ds']     = test_df['ds'].values
    return result, m, forecast

prophet_result, prophet_model, prophet_forecast = run_prophet(train_total, test_total, 'total')

Prophet is configured with multiplicative seasonality — federal spending grows over time, so the seasonal swings should scale with the level, not stay fixed in dollar terms. The `changepoint_prior_scale=0.05` keeps the trend flexible but not overly wiggly. The COVID regressor is set to 1 for the years when the federal stimulus money was active in real calendar terms (fiscal Q1 2020 falls in Oct 2019 on the calendar). `QS-OCT` is the quarterly frequency starting in October — matching the federal fiscal year.

In [ ]:
# Plot Prophet forecast vs actuals
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(total['ds'], total['obligated_amount'], 'o-', ms=4, lw=1.5, label='Actual', color='steelblue')

train_forecast = prophet_forecast.iloc[:len(train_total)]
test_forecast  = prophet_forecast.iloc[len(train_total):]

ax.fill_between(prophet_forecast['ds'],
                prophet_forecast['yhat_lower'], prophet_forecast['yhat_upper'],
                alpha=0.15, color='orange', label='95% confidence')
ax.plot(prophet_forecast['ds'], prophet_forecast['yhat'], '--', lw=1.5,
        color='orange', label='Prophet forecast')

ax.axvline(pd.Timestamp('2023-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Prophet — Total Federal Spending Forecast')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f'\nProphet test period predictions vs actuals:')
comparison = pd.DataFrame({
    'Quarter': [f'FY{r.fy} Q{r.quarter}' for _, r in test_total.iterrows()],
    'Actual ($B)':    (test_total['obligated_amount'].values / 1e9).round(1),
    'Prophet ($B)':   (prophet_result['pred'] / 1e9).round(1),
    'Error ($B)':     ((prophet_result['pred'] - test_total['obligated_amount'].values) / 1e9).round(1)
})
print(comparison.to_string(index=False))

**Prophet Results — MAPE 27.6% | MAE $1,765B | RMSE $2,188B**

Prophet consistently **underestimated** federal spending across all 8 test quarters. The gap widened sharply through FY2024: the model was $509B below actual in Q1 FY2024 and $4,553B below in Q4 FY2024 — a 48% underestimate by year-end. The root cause is a structural spending level shift after COVID that Prophet's trend flexibility couldn't fully absorb: post-2021 federal spending settled at a permanently higher baseline (driven by the Inflation Reduction Act, CHIPS Act, and expanded entitlement obligations) that is outside the trend pattern seen in the FY2017–2022 training window. The 95% confidence band did not contain the actual values in FY2024, which means the model is not just wrong on the point estimate but also underestimating its own uncertainty.

## 4. SARIMA

In [ ]:
def run_sarima(train_df, test_df, order=(1,1,1), seasonal_order=(1,1,0,4), series_name='total'):
    train_y    = train_df.set_index('ds')['obligated_amount']
    train_exog = train_df.set_index('ds')[['covid']]
    test_exog  = test_df.set_index('ds')[['covid']]

    model = SARIMAX(train_y,
                    exog=train_exog,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    fitted = model.fit(disp=False)

    pred = fitted.forecast(steps=len(test_df), exog=test_exog)
    true = test_df['obligated_amount'].values
    result = metrics(true, pred.values, label='SARIMA')
    result['series'] = series_name
    result['pred']   = pred.values
    result['true']   = true
    result['ds']     = test_df['ds'].values
    return result, fitted

sarima_result, sarima_fitted = run_sarima(train_total, test_total, series_name='total')
print(f'\nAIC: {sarima_fitted.aic:.1f}   BIC: {sarima_fitted.bic:.1f}')

SARIMAX with order (1,1,1)(1,1,0,4) — one autoregressive term, one differencing pass to remove trend, one moving average term, and a seasonal component with period 4 (quarterly). The COVID column is passed as an exogenous variable, same as in Prophet. AIC and BIC are model fit diagnostics — lower is better, and they're useful later if you want to compare different SARIMA configurations. `enforce_stationarity=False` avoids errors when the seasonal component makes the model technically non-stationary at initialization.

In [ ]:
# Plot SARIMA forecast vs actuals
pred_index = test_total['ds'].values

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(total['ds'], total['obligated_amount'], 'o-', ms=4, lw=1.5, label='Actual', color='steelblue')
ax.plot(pred_index, sarima_result['pred'], 's--', ms=5, lw=1.5, label='SARIMA forecast', color='green')
ax.axvline(pd.Timestamp('2023-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('SARIMA — Total Federal Spending Forecast')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print('\nSARIMA test period predictions vs actuals:')
comparison_s = pd.DataFrame({
    'Quarter':        [f'FY{r.fy} Q{r.quarter}' for _, r in test_total.iterrows()],
    'Actual ($B)':    (test_total['obligated_amount'].values / 1e9).round(1),
    'SARIMA ($B)':    (sarima_result['pred'] / 1e9).round(1),
    'Error ($B)':     ((sarima_result['pred'] - test_total['obligated_amount'].values) / 1e9).round(1)
})
print(comparison_s.to_string(index=False))

**SARIMA Results — MAPE 3.8% | MAE $144B | RMSE $157B | AIC 759.4**

SARIMA is the clear winner on the total series. With only 3.8% MAPE and errors balanced on both sides (FY2023 Q2 was -$233B under, FY2024 Q2 was +$182B over), the model did not systematically drift in either direction. The worst single-quarter miss was Q1 FY2023 at +$150B — well under 7% of the actual value. The (1,1,1)(1,1,0,4) structure works well here: the double differencing (one regular + one seasonal) removes both the year-over-year trend and the within-year accumulation pattern, leaving a stationary residual that the AR and MA terms handle cleanly. The low AIC (759.4) and BIC (762.3) confirm the model complexity is appropriate for this dataset size.

## 5. XGBoost

In [ ]:
# Build features for XGBoost across ALL budget functions
df_xgb = df.sort_values(['budget_function_id','ds']).copy()

# Lag features within each function
df_xgb['lag_1'] = df_xgb.groupby('budget_function_id')['obligated_amount'].shift(1)
df_xgb['lag_4'] = df_xgb.groupby('budget_function_id')['obligated_amount'].shift(4)
df_xgb['lag_8'] = df_xgb.groupby('budget_function_id')['obligated_amount'].shift(8)
df_xgb['roll4_mean'] = df_xgb.groupby('budget_function_id')['obligated_amount'].transform(
    lambda x: x.shift(1).rolling(4).mean())

# Encode function id
le = LabelEncoder()
df_xgb['func_enc'] = le.fit_transform(df_xgb['budget_function_id'])

df_xgb = df_xgb.dropna(subset=['lag_1','lag_4','lag_8'])

FEATURES = ['func_enc','fy','quarter','quarter_sin','quarter_cos','covid',
            'lag_1','lag_4','lag_8','roll4_mean']
TARGET   = 'obligated_amount'

train_xgb = df_xgb[df_xgb['fy'] <= TRAIN_END]
test_xgb  = df_xgb[df_xgb['fy'] >= TEST_START]

print(f'XGBoost train: {len(train_xgb):,} rows   test: {len(test_xgb):,} rows')
print(f'Features: {FEATURES}')

XGBoost needs structured features — it doesn't understand time series as a sequence the way SARIMA does. Instead it learns from lag values: what was this function's spending one quarter ago, four quarters ago (same quarter last year), and eight quarters ago (two years back). The rolling 4-quarter mean gives it a smoothed recent trend. The function ID is label-encoded so XGBoost can use it as a categorical feature — one model handles all 20 budget functions at once.

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(train_xgb[FEATURES], train_xgb[TARGET],
              eval_set=[(test_xgb[FEATURES], test_xgb[TARGET])],
              verbose=False)

xgb_pred = xgb_model.predict(test_xgb[FEATURES])
xgb_true = test_xgb[TARGET].values
xgb_result = metrics(xgb_true, xgb_pred, label='XGBoost')
xgb_result['series'] = 'all_functions'

# Feature importance
importance = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\nFeature importance:')
print(importance.round(4).to_string())

**XGBoost Feature Importance — lag_4 dominates at 43.2%**

The most important feature is `lag_4` (same quarter one year ago) at 43.2% importance, followed by `lag_8` (same quarter two years ago) at 27.3%. Together these two year-ago references account for 70% of the model's decision weight — confirming that federal spending is highly cyclical and the strongest signal is simply what happened at the same point in prior fiscal years. The current quarter's `lag_1` (9.1%) and `roll4_mean` (7.4%) add marginal improvement. The `covid` flag contributes just 1.1%, meaning the model found adequate signal in the lag values alone to handle the COVID period — the lag features from FY2020–2021 implicitly carry the COVID spending information forward.

In [ ]:
# XGBoost: plot total spending (sum across functions per quarter in test set)
test_xgb_copy = test_xgb.copy()
test_xgb_copy['xgb_pred'] = xgb_pred

xgb_total_pred = test_xgb_copy.groupby('ds')[['obligated_amount','xgb_pred']].sum().reset_index()
train_actual   = df_xgb[df_xgb['fy'] <= TRAIN_END].groupby('ds')['obligated_amount'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_actual['ds'], train_actual['obligated_amount'], 'o-', ms=3, lw=1.5,
        color='steelblue', label='Actual (train)')
ax.plot(xgb_total_pred['ds'], xgb_total_pred['obligated_amount'], 'o-', ms=3, lw=1.5,
        color='steelblue', alpha=0.4, label='Actual (test)')
ax.plot(xgb_total_pred['ds'], xgb_total_pred['xgb_pred'], 's--', ms=5, lw=1.5,
        color='purple', label='XGBoost forecast')
ax.axvline(pd.Timestamp('2023-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('XGBoost — Total Federal Spending Forecast (Sum Across Functions)')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

xgb_metrics_total = metrics(xgb_total_pred['obligated_amount'].values,
                            xgb_total_pred['xgb_pred'].values, label='XGB-Total')

**XGBoost Total (Summed) — MAPE 8.3% | MAE $281B | RMSE $335B**

Summing XGBoost's per-function predictions gives a total series MAPE of 8.3% — better than Prophet (27.6%) but worse than SARIMA (3.8%). The bottom-up aggregation works reasonably well here: the individual function errors partially cancel each other out when summed. XGBoost's advantage is that it produces function-level forecasts and a total with one trained model, whereas SARIMA needs a separate fit per series. For functions with strong year-over-year regularity (Medicare, Social Security, Defense), XGBoost's lag features give it a solid foundation; for volatile functions (Commerce & Housing Credit, Income Security), it benefits from borrowing signal across functions through the shared model.

## 6. Per-Function Forecasts (Prophet + SARIMA)

In [ ]:
# Run Prophet and SARIMA for each top-10 function
func_results = []

for fid in top10_ids:
    fname = func_names.get(fid, fid)
    sub   = df[df['budget_function_id'] == fid].sort_values('ds')
    tr    = sub[sub['fy'] <= TRAIN_END]
    te    = sub[sub['fy'] >= TEST_START]

    if len(tr) < 8 or len(te) == 0:
        continue

    # Prophet
    try:
        r_p, _, _ = run_prophet(tr, te, series_name=fid)
        r_p['function_name'] = fname
        func_results.append(r_p)
    except Exception as e:
        print(f'  Prophet failed for {fid}: {e}')

    # SARIMA
    try:
        r_s, _ = run_sarima(tr, te, series_name=fid)
        r_s['function_name'] = fname
        func_results.append(r_s)
    except Exception as e:
        print(f'  SARIMA failed for {fid}: {e}')

print(f'\nCompleted {len(func_results)} model runs across {len(top10_ids)} functions.')

This loops through the top 10 budget functions and fits both Prophet and SARIMA to each one independently. The try/except blocks are essential here — SARIMA numerically diverged on 3 of 10 functions (Income Security, Social Security, Commerce & Housing Credit) due to coefficient explosion after seasonal differencing on highly volatile series. Those failures are caught and printed so the pipeline continues. Prophet completed successfully on all 10 functions, making it the more robust option for per-function deployment.

In [ ]:
# Summary table of per-function results
rows = []
for r in func_results:
    rows.append({
        'Function':  r.get('function_name', r['series'])[:35],
        'Model':     r['model'],
        'MAE ($B)':  round(r['MAE'] / 1e9, 2),
        'RMSE ($B)': round(r['RMSE'] / 1e9, 2),
        'MAPE (%)':  round(r['MAPE'], 1)
    })

results_df = pd.DataFrame(rows)
print(results_df.sort_values(['Function','Model']).to_string(index=False))

**Per-Function Results — Prophet more reliable; SARIMA diverged on 3 functions**

**Prophet per-function highlights (MAPE):**
- Social Security: **3.0%** — best overall
- Veterans Benefits: **4.5%**
- National Defense: **6.6%**
- Medicare: **6.5%**
- Health: **8.0%**
- Net Interest: **12.2%**
- General Government: **24.2%**
- Education: **74.9%** — COVID-era fluctuations in stimulus spending
- Income Security: **147.1%** — highly volatile due to pandemic relief programs
- Commerce & Housing Credit: **250.8%** — swings from FDIC/bank interventions

**SARIMA per-function failures:** SARIMA diverged numerically on **Income Security** (MAPE 423 million %), **Social Security** (MAPE 172,000%), and **Commerce & Housing Credit** (MAPE 1,124%). These functions have spending patterns that violate SARIMA's stationarity assumptions after differencing — particularly Income Security, where pandemic relief injections created an impulse that the AR/MA terms cannot represent without explosive coefficients. For these functions, Prophet or XGBoost should be the model of choice in the dashboard.

In [ ]:
# Plot forecast vs actual for top 4 functions (2x2 grid)
top4 = top10_ids[:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, fid in zip(axes.flat, top4):
    fname = func_names.get(fid, fid)
    sub   = df[df['budget_function_id'] == fid].sort_values('ds')

    # actual line
    ax.plot(sub['ds'], sub['obligated_amount'], 'o-', ms=3, lw=1.5,
            label='Actual', color='steelblue')

    # prophet prediction
    p_res = next((r for r in func_results if r['series'] == fid and r['model'] == 'Prophet'), None)
    s_res = next((r for r in func_results if r['series'] == fid and r['model'] == 'SARIMA'), None)

    if p_res:
        ax.plot(p_res['ds'], p_res['pred'], 's--', ms=4, lw=1.2,
                label='Prophet', color='orange')
    if s_res:
        ax.plot(s_res['ds'], s_res['pred'], '^--', ms=4, lw=1.2,
                label='SARIMA', color='green')

    ax.axvline(pd.Timestamp('2023-10-01'), color='red', lw=0.8, ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(billions))
    ax.set_title(f'{fid} — {fname[:30]}', fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Prophet vs SARIMA — Top 4 Budget Functions (Test Period)', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**Top 4 functions — Medicare, Social Security, National Defense, Income Security**

- **Medicare** (570): Both Prophet (6.5% MAPE) and SARIMA (13.3% MAPE) tracked well. Medicare spending is highly predictable — legislated reimbursement rates grow on a known schedule. Prophet was the better model here.
- **Social Security** (650): Prophet achieved 3.0% MAPE — the most accurate individual function result in the entire analysis. SARIMA diverged numerically on this series (coefficient explosion). Prophet clearly wins for Social Security.
- **National Defense** (050): Prophet (6.6%) and SARIMA (12.0%) both produced reasonable forecasts. Defense appropriations are set annually through Congress, giving them strong year-over-year regularity that both models could exploit.
- **Income Security** (600): Both models struggled. Prophet reached 147.1% MAPE and SARIMA diverged completely. This function absorbed the bulk of pandemic stimulus (CARES Act, American Rescue Plan), making FY2020–2021 extreme outliers that distort forecasts even with the COVID flag in place.

## 7. Model Comparison

In [ ]:
# Side-by-side comparison: Prophet vs SARIMA vs XGBoost on total series
total_comparison = pd.DataFrame([
    {'Model': 'Prophet',  'MAE ($B)': round(prophet_result['MAE']/1e9, 2),
     'RMSE ($B)': round(prophet_result['RMSE']/1e9, 2), 'MAPE (%)': round(prophet_result['MAPE'], 1)},
    {'Model': 'SARIMA',   'MAE ($B)': round(sarima_result['MAE']/1e9, 2),
     'RMSE ($B)': round(sarima_result['RMSE']/1e9, 2),  'MAPE (%)': round(sarima_result['MAPE'], 1)},
    {'Model': 'XGBoost',  'MAE ($B)': round(xgb_metrics_total['MAE']/1e9, 2),
     'RMSE ($B)': round(xgb_metrics_total['RMSE']/1e9, 2), 'MAPE (%)': round(xgb_metrics_total['MAPE'], 1)},
])
print('=== Total Federal Spending — Model Comparison ===')
print(total_comparison.to_string(index=False))

# Bar chart of MAPE
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['orange', 'green', 'purple']
for ax, col in zip(axes, ['MAE ($B)', 'RMSE ($B)', 'MAPE (%)']):
    ax.bar(total_comparison['Model'], total_comparison[col], color=colors)
    ax.set_title(col)
    for i, v in enumerate(total_comparison[col]):
        ax.text(i, v * 1.01, str(v), ha='center', fontsize=9)
plt.suptitle('Model Comparison — Total Federal Spending', y=1.02)
plt.tight_layout()
plt.show()

**Final Scorecard — SARIMA wins the total series; Prophet wins per-function**

| Model   | MAE      | RMSE     | MAPE   |
|---------|----------|----------|--------|
| SARIMA  | $144B    | $157B    | **3.8%** ✓ |
| XGBoost | $281B    | $335B    | 8.3%   |
| Prophet | $1,765B  | $2,188B  | 27.6%  |

**On the total federal spending series**, SARIMA outperforms the other two models by a wide margin — nearly 7× better than XGBoost and 7× better than Prophet on MAPE. Its autoregressive structure naturally captures the cumulative within-year spending accumulation pattern that characterizes USASpending.gov data.

**Per-function**, the picture flips: SARIMA diverged on 3 of the 10 functions (Income Security, Social Security, Commerce & Housing Credit), while Prophet remained numerically stable across all 10 with reasonable accuracy on mandatory spending programs. For the dashboard, the recommended strategy is: **use SARIMA for the total series** and **use Prophet as the primary per-function model** (with XGBoost as backup for high-volatility functions).

## 8. Save Forecasts

In [ ]:
# Save all results to forecasts folder for dashboard use

# 1. Total series predictions
total_preds = pd.DataFrame({
    'ds':        test_total['ds'].values,
    'fy':        test_total['fy'].values,
    'quarter':   test_total['quarter'].values,
    'actual':    test_total['obligated_amount'].values,
    'prophet':   prophet_result['pred'],
    'sarima':    sarima_result['pred'],
    'xgboost':   xgb_total_pred['xgb_pred'].values,
})
total_preds.to_csv(FORECAST_DIR / 'budget_functions_total_predictions.csv', index=False)

# 2. Per-function predictions
func_pred_rows = []
for r in func_results:
    for i, (ds, pred, true) in enumerate(zip(r['ds'], r['pred'], r['true'])):
        func_pred_rows.append({
            'ds': ds, 'budget_function_id': r['series'],
            'function_name': r.get('function_name', ''),
            'model': r['model'], 'actual': true, 'predicted': pred
        })
func_preds = pd.DataFrame(func_pred_rows)
func_preds.to_csv(FORECAST_DIR / 'budget_functions_per_function_predictions.csv', index=False)

# 3. Model metrics summary
total_comparison.to_csv(FORECAST_DIR / 'budget_functions_model_metrics.csv', index=False)

print('Saved to', FORECAST_DIR)
for f in sorted(FORECAST_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

Three files saved to `data/forecasts/`: the total series predictions with all three models side by side, the per-function predictions for the top 10 functions, and the metrics summary table. The dashboard will load these CSVs directly — no need to re-run the models every time the dashboard starts, which would be too slow for an interactive app.